# Submission 1: Your Model, Your Beam
### ME 323 Module 1

<img src="https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/me323/Module1_drafts/figures/I_beam_dimensions.jpg" alt="I-beam dimensions" width="220">

Both common class designs have been queried. The scoreboard so far:

| design | (b, H_web) | predicted | returned strength and str/w on estimated-mass basis |
|---|---|---|---|
| equation-query beam from Pre-lab 1 | (1.10, 13.25) | 48.7 N/g | **37.48 N/g** (475.7 N) |
| locked GP-query beam from Pre-lab 2, MUI ψ=1 | (1.00, 13.39) | 36.7 N/g | **36.65 N/g** (445.8 N) |

The equation beam beat the 15-beam handout best of 36.88 N/g and is the best
of all 17. The GP beam probed the b = 1.0 edge of the newly opened box and
came back at 36.65 N/g — below the handout best, and close to what the model
expected there. Relative to their own predictions, the equation result came
in 23.0% low and the GP result just 0.0% below its 36.7 N/g prediction. Sit
with that pair of facts: the winning beam came from the prediction that
missed by 23.0%, while the nearly calibrated prediction described a beam not
worth building. A model can be honest about a mediocre region and still lose
to an overconfident one that happened to point somewhere better. One test
settles less than it seems to.

Both returned strengths came from the staff ground truth model — fit to the
prior test campaign, standing in for the testing machine — so nothing has
been printed yet. That changes here: fold both query results into the data,
build a model your way, and commit to a third design, the beam your group will
actually print and break. Do not call either common query beam your
Submission 1 design unless you deliberately choose those coordinates. The
final answer is a decision defended in the memo.

## Your knobs

A **knob** is a variable or setting you deliberately change to test a modeling
assumption or decision rule. This notebook exposes five:

| knob | variable | choices | what changes |
|---|---|---|---|
| model architecture | `CHOICE` | A plain / B strength model / C features / D residual error | what the GP sees and predicts |
| kernel | `KERNEL` | RBF / Matérn | assumed smoothness |
| assumed noise | `NOISE_PCT` | percent; 3% class default | how tightly the GP follows individual tests |
| acquisition rule | `ACQ` | MUI / EI | how value and uncertainty form a score |
| score dial | `psi` or `xi` | chosen for the selected rule | how uncertainty affects the recommendation |

Architecture and the acquisition rule are design decisions. Kernel and noise
are assumptions to check after selecting a candidate. The target quantity is
part of the architecture rather than a separate sixth choice.

Record the exact knob values used for every reported prediction. Changing a
knob requires refitting the model; a length scale, median, uncertainty, or
recommendation from one setting should not be presented as if it came from
another. Sections 2–4 provide the evidence needed to choose and check them.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120

# Fixed geometry (you choose b and H_web; everything else is set)
B, TH, L = 10.0, 18.0, 150.0     # flange width, total height, test span (mm)
LP = 172.0                        # printed length (mm); overhangs the 150 mm span
KMASS = 0.2045                    # g/mm^2: mass per unit cross-section area at 172 mm

# Handbook starting values — Pre-lab 1 calibrates the three marked ones
SY = 76e6                         # Pa, PLA strength                 (calibrated)
K_LTB = 0.33                      # fixture effective-length factor  (calibrated)
TAU_I = 43.9e6                    # printed-interface shear strength (Pa) — starting
                                  # guess = bulk yield / sqrt(3)     (calibrated)
E, G = 2.5e9, 2.5e9 / 2.6         # Young's / shear modulus (Pa) — fixed
C1, C2 = 1.35, 0.55               # LTB moment-gradient / load-height factors
URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
       "core-me-data-science-activities-public/main/data/student_beams_B10_L150.csv")
try:
    df = pd.read_csv(URL); print("loaded from GitHub")
except Exception:
    try:
        df = pd.read_csv("student_beams_B10_L150.csv"); print("loaded local copy")
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "no internet and no local copy -- download student_beams_B10_L150.csv "
            "from the course page into this notebook's folder and rerun") from e
df = df.rename(columns={"b_mm": "b", "H_web_mm": "H"})

def estimated_mass_g(b, H):
    A = b * H + B * (TH - H)      # cross-section area, mm^2
    return KMASS * A              # grams
df["mass_est_g"] = estimated_mass_g(df.b, df.H)
df["mass_delta_g"] = df.weight_g - df.mass_est_g
df["mass_delta_pct"] = 100*df.mass_delta_g/df.mass_est_g
print(len(df), "tested beams")

new = pd.DataFrame([
    dict(beam_id=16, b=1.10, H=13.25, strength_N=475.7,
         failure_note="equation-query result; observed morphology not supplied"),
    dict(beam_id=17, b=1.00, H=13.39, strength_N=445.8,
         failure_note="locked-GP-query result; observed morphology not supplied"),
])
new["weight_g"] = np.nan
new["mass_est_g"] = estimated_mass_g(new.b, new.H)
df = pd.concat([df, new], ignore_index=True)
df["str_to_weight"] = df.strength_N / df.mass_est_g
print(len(df), "beams. best observed:", round(df.str_to_weight.max(), 2), "N/g")
pd.set_option("display.max_colwidth", None)
print("\nFailure-note evidence available to the design decision:")
print(df[["beam_id", "b", "H", "failure_note"]].to_string(index=False))

## 1. The calibrated physics (carried over from Pre-lab 1)

Provided complete this time, with the class calibration values baked in.

In [ ]:
def section_props(b, H):
    """I-section properties. b, H in mm; everything returned in METERS/SI."""
    tf = (TH - H) / 2.0
    b_, h_, B_, tf_ = b/1e3, H/1e3, B/1e3, tf/1e3
    c = (TH/1e3) / 2
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12
    def J_rect(x, y):
        short, long = min(x, y), max(x, y)
        r = short/long
        beta = 1 - 0.63*r + 0.052*r**5
        return (1/3)*beta*long*short**3
    J = J_rect(b_, h_) + 2*J_rect(tf_, B_)
    Cw = Iy*(h_ + tf_)**2/4
    return dict(Ix=Ix, Iy=Iy, J=J, Cw=Cw, c=c,
                b=b_, h=h_, tf=tf_, B=B_)

def P_bend(p, sy):
    return 4*sy*p["Ix"] / (p["c"] * L/1e3)
def P_LTB(p, sy, k):
    My = sy*p["Ix"]/p["c"]
    Lb, zg = k*L/1e3, p["c"]
    R = p["Cw"]/p["Iy"] + (Lb**2*G*p["J"])/(np.pi**2*E*p["Iy"]) + (C2*zg)**2
    Mcr = C1*np.pi**2*E*p["Iy"]/Lb**2 * (np.sqrt(R) - C2*zg)
    return 4*min(My, Mcr)/(L/1e3)
def Q_flange(p):
    return p["B"]*p["tf"]*(p["h"]/2 + p["tf"]/2)
def P_sep(p, tau_i):
    """Flange-web separation: shear flow vs strength along the printed layer lines."""
    return 2*tau_i*p["Ix"]*p["b"]/Q_flange(p)
def capacity(b, H, sy, k, tau_i):
    """Class model (2026-07-15): plain minimum of the three mode capacities."""
    p = section_props(b, H)
    return min(P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k))
def gov_mode(b, H, sy, k, tau_i):
    """Dominant pure-mode proxy, not an observed failure-mechanism label."""
    p = section_props(b, H)
    Pb, Ps, Pl = P_bend(p, sy), P_sep(p, tau_i), P_LTB(p, sy, k)
    return "separation" if Ps < min(Pb, Pl) else (
        "LTB" if Pl < 0.999*Pb else "bend")

SY_CAL, K_CAL, TAU_CAL = 6.683e+07, 0.377, 1.676e+07   # your Pre-lab 1 calibration
print("calibrated capacity(2,12) =", round(capacity(2, 12, SY_CAL, K_CAL, TAU_CAL), 1), "N")

## 2. Compare four model architectures

"Model architecture" here means two choices at once: what the GP sees, and
what quantity it learns.

| model | what the GP sees | what it predicts |
|---|---|---|
| **A — plain data-only GP** | (b, H_web) | log str/w |
| **B — strength (not str/w) model** | (b, H_web) | log strength; divide by mass afterward |
| **C — physics-feature model** | (b, H_web, log P_phys, P_LTB/P_bend) | log str/w |
| **D — learn residual error from physics predictions** | (b, H_web) | log(measured / P_phys) |

What each model assumes:

- **A — plain (vanilla GP).** A pure pattern-matcher: str/w is a smooth
  function of the two geometry variables. It does not use the capacity
  equations.
- **B — strength (not str/w) model.** The GP learns raw strength in newtons;
  estimated mass is applied afterward. This is appropriate only if strength
  is the more learnable surface for the final strength-to-weight decision.
- **C — physics features.** The GP still learns str/w, but you hand it two
  extra inputs computed from the calibrated equations: the physics capacity
  and an LTB-to-bending ratio. Those features remain part of the GP distance
  calculation everywhere. If they encode incomplete physics, they can distort
  predictions; the model does not automatically discard them.
- **D — learn residual error from physics predictions.** The equations supply
  a baseline and the GP learns the multiplicative error in that baseline.
  This carries the physics surface into predictions between tests, but it also
  carries every omitted mechanism and calibration limitation.

The leave-one-out (LOO) table predicts each beam without that beam in the fit,
except for learning the 3 physics constants. Lower RMSE supports a model on
these 17 locations. Small differences should not be overinterpreted, and LOO
does not establish behavior outside the tested region. Compare the table with
the failure notes and the location of your proposed design.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, Matern

def make_kernel(kernel, ndim):
    """The smoothness assumption, written as code.
    RBF:    infinitely differentiable — believes the strength surface is
            gentle everywhere; can round over a sharp failure-mode handoff.
    Matern: nu = 2.5, twice differentiable — believes the surface may carry
            kinks, e.g. where the governing failure mode changes."""
    if kernel == "RBF":
        return C(1.0, (1e-3, 1e3)) * RBF([1.0]*ndim, (1e-1, 30.0))
    elif kernel == "Matern":
        return C(1.0, (1e-3, 1e3)) * Matern([1.0]*ndim, (1e-1, 30.0), nu=2.5)
    raise ValueError(f"unknown kernel {kernel!r}: use 'RBF' or 'Matern'")

def fit_gp(data, alpha=0.03**2, feats=("b", "H"), target="log_sw", kernel="RBF"):
    """The class GP recipe: z-scored inputs, log target, MLE hyperparameters."""
    X = data[list(feats)].values.astype(float)
    fmu, fsd = X.mean(0), X.std(0) + 1e-12
    if target == "log_sw":
        y = np.log(data.strength_N.values /
                   estimated_mass_g(data.b.values, data.H.values))
    elif target == "log_strength":
        y = np.log(data.strength_N.values.astype(float))
    else:                                    # log residual vs calibrated physics
        Pphys = np.array([capacity(b, H, SY_CAL, K_CAL, TAU_CAL)
                          for b, H in zip(data.b, data.H)])
        y = np.log(data.strength_N.values) - np.log(Pphys)
    ymean = y.mean()
    gp = GaussianProcessRegressor(make_kernel(kernel, X.shape[1]), alpha=alpha,
                                  normalize_y=False,
                                  n_restarts_optimizer=5, random_state=0)
    gp.fit((X - fmu) / fsd, y - ymean)
    return gp, fmu, fsd, ymean

def predict_sw(gp, fmu, fsd, ymean, bq, Hq, target, feats):
    Xq = build_feats(bq, Hq, feats)
    mu, sd = gp.predict((Xq - fmu)/fsd, return_std=True)
    mu = mu + ymean
    mass = estimated_mass_g(np.asarray(bq, float), np.asarray(Hq, float))
    if target == "log_sw":
        sw = np.exp(mu)
    elif target == "log_strength":
        sw = np.exp(mu)/mass
    else:
        Pphys = np.array([capacity(b, H, SY_CAL, K_CAL, TAU_CAL)
                          for b, H in zip(np.atleast_1d(bq), np.atleast_1d(Hq))])
        sw = Pphys*np.exp(mu)/mass
    return sw, sd

def build_feats(bq, Hq, feats):
    bq, Hq = np.atleast_1d(np.asarray(bq, float)), np.atleast_1d(np.asarray(Hq, float))
    cols = {"b": bq, "H": Hq}
    if "logP" in feats or "stab" in feats:
        pp = [section_props(b, H) for b, H in zip(bq, Hq)]
        Pb = np.array([P_bend(p, SY_CAL) for p in pp])
        Pl = np.array([P_LTB(p, SY_CAL, K_CAL) for p in pp])
        Ps = np.array([P_sep(p, TAU_CAL) for p in pp])
        cols["logP"] = np.log(np.minimum(Pb, np.minimum(Ps, Pl)))
        cols["stab"] = Pl/Pb
    return np.column_stack([cols[f] for f in feats])

LANES = {
    "A plain":    dict(feats=("b", "H"), target="log_sw"),
    "B strength model": dict(feats=("b", "H"), target="log_strength"),
    "C features": dict(feats=("b", "H", "logP", "stab"), target="log_sw"),
    "D residual error": dict(feats=("b", "H"), target="log_residual"),
}

def fit_lane(data, lane, alpha=0.03**2, kernel="RBF"):
    cfg = LANES[lane]
    d2 = data.copy()
    Xf = build_feats(d2.b.values, d2.H.values, cfg["feats"])
    for j, f in enumerate(cfg["feats"]):
        d2[f] = Xf[:, j]
    gp, fmu, fsd, ymean = fit_gp(d2, alpha=alpha, feats=cfg["feats"],
                                 target=cfg["target"], kernel=kernel)
    return gp, fmu, fsd, ymean, cfg

print("leave-one-out RMSE in str/w space (lower is better):")
for kernel in ("RBF", "Matern"):
    print(f"  kernel = {kernel}")
    for lane in LANES:
        errs = []
        for i in range(len(df)):
            tr = df.drop(df.index[i])
            gp_, fmu_, fsd_, ym_, cfg_ = fit_lane(tr, lane, kernel=kernel)
            sw_hat, _ = predict_sw(gp_, fmu_, fsd_, ym_, df.b.iloc[i], df.H.iloc[i],
                                   cfg_["target"], cfg_["feats"])
            errs.append(sw_hat[0] - df.str_to_weight.iloc[i])
        print(f"    {lane}: {np.sqrt(np.mean(np.array(errs)**2)):.2f} N/g")
print("\nCHECKPOINT (RBF):    roughly A plain 2.94,  B strength model 6.35,  C features 3.07,  D residual error 2.38 N/g.")
print("CHECKPOINT (Matern): roughly A plain 2.89,  B strength model 5.40,  C features 3.13,  D residual error 2.34 N/g.")
print("Within about 0.1 N/g of these counts as matching -- library versions")
print("wobble the fits. Off by 0.5 N/g or more, check your work or talk to a TA.")

## 3. Rebuild the maps with your knobs

Set `CHOICE`, `KERNEL`, and `NOISE_PCT`, then refit on all 17 beams.

- **Kernel:** RBF assumes a very smooth surface; Matérn-5/2 allows sharper
  changes. Compare their LOO errors and remember that failure-mode handoffs
  can create slope changes near attractive designs.
- **Noise:** `alpha=(NOISE_PCT/100)**2` is the assumed repeat-to-repeat
  variance in log space. Smaller values follow individual observations more
  closely; larger values smooth more. The class uses 3%, while pooled campaign
  repeats are closer to 4.4% with session structure.

No single fit establishes the correct kernel or noise. After choosing a
candidate, repeat the fit under the other kernel and under 1%, 3%, and 10%
noise, then report whether the recommendation moves.

In [ ]:
CHOICE    = "A plain"   # <<< model: "A plain" | "B strength model" | "C features" | "D residual error"
KERNEL    = "RBF"       # <<< your smoothness assumption: "RBF" | "Matern"
NOISE_PCT = 3           # <<< your assumed repeatability, percent

gpF, fmuF, fsdF, ymF, cfgF = fit_lane(df, CHOICE, alpha=(NOISE_PCT/100)**2,
                                      kernel=KERNEL)
bg = np.linspace(1.0, 7.0, 63); Hg = np.linspace(5.0, 16.0, 60)
BB, HH = np.meshgrid(bg, Hg)
SWg, SDg = predict_sw(gpF, fmuF, fsdF, ymF, BB.ravel(), HH.ravel(),
                      cfgF["target"], cfgF["feats"])
MU, STD = SWg.reshape(BB.shape), SDg.reshape(BB.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for a, Z, t in [(ax[0], MU, f"posterior median str/w: {CHOICE}, {KERNEL}"),
                (ax[1], STD, "epistemic uncertainty (log units)")]:
    cf = a.contourf(BB, HH, Z, levels=20); fig.colorbar(cf, ax=a)
    a.scatter(df.b, df.H, c="w", edgecolor="k", s=40)
    a.scatter(df.b.iloc[-2:], df.H.iloc[-2:], c="red", marker="*", s=170,
              label="the two class beams")
    a.set_xlabel("b [mm]"); a.set_ylabel("H_web [mm]"); a.set_title(t); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

### The decision map: four panels, one argument

Before choosing, put the module's two models side by side on the same axes.
The four panels below are the figure most worth reading in this notebook —
they turn "which model do we trust *here*" into something you can point at:

1. **Calibrated physics** — predicted str/w with the governing-mode boundaries
   drawn on. Where the boundaries run is where the physics rests on its most
   fragile assumptions.
2. **Your GP posterior median** — what the data support, under your knobs.
3. **Physics minus GP** — the disagreement map. Near-zero where the two
   stories agree; large where at least one of them is wrong.
4. **Epistemic sigma** — where the GP is guessing.

Read your candidate against all four: Do the models agree there? Is the
agreement backed by nearby tests, or is it two extrapolations shaking hands?
Does the disagreement track a mode boundary? Is the attractive region
promising, or merely unexplored? These are card rows 6 and 7, drawn
instead of written.

In [ ]:
from matplotlib.lines import Line2D

PHYS = np.array([capacity(b, H, SY_CAL, K_CAL, TAU_CAL) /
                 estimated_mass_g(b, H)
                 for b, H in zip(BB.ravel(), HH.ravel())]).reshape(BB.shape)
MODE = np.array([{"bend": 0, "separation": 1, "LTB": 2}[
                 gov_mode(b, H, SY_CAL, K_CAL, TAU_CAL)]
                 for b, H in zip(BB.ravel(), HH.ravel())]).reshape(BB.shape)

fig, ax = plt.subplots(2, 2, figsize=(12, 8.6))
panels = [(ax[0, 0], PHYS, "calibrated physics str/w [N/g] + mode boundaries", "viridis"),
          (ax[0, 1], MU, f"GP posterior median str/w [N/g]  ({CHOICE}, {KERNEL})", "viridis"),
          (ax[1, 0], PHYS - MU, "disagreement: physics minus GP [N/g]", "coolwarm"),
          (ax[1, 1], STD, "epistemic sigma (log units)", "viridis")]
vmax = float(np.abs(PHYS - MU).max())
for a, Z, t, cm in panels:
    kw = dict(levels=20, cmap=cm)
    if cm == "coolwarm": kw.update(vmin=-vmax, vmax=vmax)
    cf = a.contourf(BB, HH, Z, **kw); fig.colorbar(cf, ax=a)
    a.contour(BB, HH, MODE, levels=[0.5, 1.5], colors="k", linewidths=0.8)
    a.scatter(df.b.iloc[:-2], df.H.iloc[:-2], c="w", edgecolor="k", s=32)
    a.scatter(df.b.iloc[-2:], df.H.iloc[-2:], c="red", marker="*", s=150,
              clip_on=False)
    a.set_xlabel("b [mm]"); a.set_ylabel("H_web [mm]"); a.set_title(t, fontsize=10)
legend_items = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="white",
           markeredgecolor="black", label="tested beams"),
    Line2D([0], [0], marker="*", color="none", markerfacecolor="red",
           markeredgecolor="red", markersize=12, label="class-query beams"),
    Line2D([0], [0], color="black", lw=1, label="physics mode boundaries"),
]
fig.legend(handles=legend_items, loc="lower center", ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()


## 4. Choose a score

The model supplies `MU`, the posterior median str/w, and `STD`, epistemic
uncertainty in log units. Choose how they become a score:

- **MUI:** `log(MU) + psi*STD`. At `psi=0` it selects the highest median;
  larger `psi` gives uncertain designs more weight.
- **EI:** expected improvement over the best tested beam (37.48 N/g).
  `xi` is the additional improvement margin. Increasing ξ often favors
  less-certain regions, but does not guarantee a more exploratory result.

MUI applies the same sigma bonus everywhere. EI weights possible gains by
their posterior probabilities. State the rule and dial that match your risk
posture, then compute the recommendation.

### Inspect a boundary recommendation

A boundary recommendation means the score was still increasing when the
search reached the edge of the allowed box. Evidence is one-sided there, and
a corner is bounded in two directions. Report the distance to the boundary,
nearby tested beams, and whether the recommended region contains known
failure observations.

### Compare with physics

At the recommended coordinates, read the calibrated physics prediction,
governing-mode proxy, physics–GP disagreement, and failure notes. Agreement
is supporting evidence, not independent validation, because both models use
the same campaign. Disagreement requires a stated reason for prioritizing one
source.

### Commit and record

The default A-plain, RBF, 3% noise, MUI ψ=1 recipe lands near
**(b ≈ 1.39, H_web ≈ 14.88)**. Treat it as a reference, not a required
answer. Record your model, kernel, noise, score rule, dial, coordinates,
posterior median, epistemic uncertainty, and calibrated-physics prediction.

If the recommendation is within |Δb| < 0.15 mm and |ΔH_web| < 0.30 mm of a
tested beam, state whether the repeat is intended to check repeatability,
confirm an unexpectedly strong result, or serve another specific purpose.

In [ ]:
from scipy.stats import norm

ACQ = "MUI"          # <<< your rule: "MUI" | "EI"
psi = 1.0            # <<< explore-exploit dial for MUI: 0 = pure exploit
xi  = 0.0            # <<< explore-exploit dial for EI: 0 = neutral; larger = more explore

if ACQ == "MUI":
    score = np.log(MU) + psi*STD              # optimistic upper bound, log space
elif ACQ == "EI":
    best = np.log(df.str_to_weight.max())     # incumbent: best tested str/w
    imp = np.log(MU) - best - xi              # win margin demanded, log space
    z = imp / STD
    score = imp*norm.cdf(z) + STD*norm.pdf(z)
else:
    raise ValueError(f"unknown ACQ {ACQ!r}: use 'MUI' or 'EI'")

i = np.unravel_index(np.argmax(score), score.shape)
b_final, H_final = float(BB[i]), float(HH[i])
print(f"FINAL DESIGN ({ACQ}):  b = {b_final:.2f} mm,  H_web = {H_final:.2f} mm")
print(f"  posterior median {MU[i]:.1f} N/g,  epistemic sigma_log {STD[i]:.3f},  "
      f"calibrated physics {capacity(b_final, H_final, SY_CAL, K_CAL, TAU_CAL)/estimated_mass_g(b_final, H_final):.1f} N/g")
print(f"  physics mode there: {gov_mode(b_final, H_final, SY_CAL, K_CAL, TAU_CAL)}")

### Optional: sweep every knob combination

Memo prompt 5 checks a handful of settings by hand. For a broader sensitivity study,
write your own code, from scratch, that loops over the combinations: four
lanes, two kernels, several noise values, both acquisition rules, and two or
three dial values. Collect each run's final coordinates into one table, save
it as a CSV, and plot where the picks land on the design box.

Read the result two ways. A region where many combinations agree is stable:
a design there does not hang on any single assumption. A knob that moves the
pick far is an assumption doing real work, and card row 4 asks you to name
it. If your recommendation holds only under your exact settings, say so in
the memo, then either move to the stable region or defend the sensitivity.

The sweep is optional, and no template is provided on purpose: the loop is
yours to design. What it produces is direct evidence for memo prompt 5 and
card row 4.

## Memo

**File the decision card first.** Fill rows 1–10 of
`ME323_Module1_DecisionCard.md` and its preregistration table, and submit them
with this notebook — the preregistration is due before your beam is printed,
and the card is reviewed before the memo. The card stays with this notebook
as the concise, on-record summary of your design decisions; the memo is where
the supporting rationale lives. The prompts below map onto card rows: the
card is the skeleton, the memo is the argument.

Write the memo as its own document, separate from this notebook, structured
as a short engineering report. Present the key outcomes of your
preregistration — the design, the predictions, the falsifiers — in the
**Results** section, then build the **Discussion** around the five prompts
below, in order, expanding the reasoning recorded on the card and citing this
notebook's figures as supporting evidence. About half a page for prompts 1–4,
a few sentences for 5. The memo covers only these prompts; the pre-lab memo
questions stay in their own notebooks.

1. The two class beams: use the generated scoreboard percentages, with their
   stated denominators. What does each miss reveal, and which miss is costlier
   for the intended design decision?
2. Your lane: what did the LOO table say under both kernels, and did you
   follow it? If you chose C or D, what does physics contribute between the
   observations? (Card rows 3 and 4.)
3. Risk: name your acquisition rule (`ACQ`, plus its dial) and your
   `NOISE_PCT`, and use the printed posterior median, epistemic sigma, and
   final coordinates to explain whether you exploited, explored, or
   replicated. Is the main risk a weak beam or a model that is confidently
   wrong? (Card row 8.)
4. Limits: flange-web separation is real in the data — beams 12, 14, and 15,
   all thin-web — and absent from all three capacity equations. Using the
   failure notes and the four-panel decision map, how close does your design
   sit to where those beams failed, and is separation a live risk for your
   beam? (Card rows 6–7, 10.)
5. Stress test: rerun sections 3 and 4 with the other kernel and at 1% and 10% noise. Did
   your coordinates survive? State either the stable-neighborhood conclusion
   or the case for using a test to reduce uncertainty instead. (Card row 4.)

The default-parameter design is already computed by the section-4 code. You do
not need a new optimization cell unless you choose a different decision rule.